# NCCT Stroke Segmentation V8: Enhanced Physics-Informed Neural Network

V7 used generic Total Variation (TV) loss — modality-agnostic, smooths all gradients equally.

V8 replaces TV with **four domain-specific physics constraints**:

| # | Loss | What it does | Why better than TV |
|---|------|-------------|-------------------|
| 1 | **ACE Loss** (Euler's Elastica) | Curvature-aware: $(\alpha + \beta \kappa^2)|\nabla u|$ | Preserves sharp edges, penalises wiggly boundaries |
| 2 | **Boundary Loss** (Kervadec) | Distance-transform boundary refinement | Class-imbalance invariant, targets boundary voxels |
| 3 | **Net Water Uptake Loss** (rHU) | CT-specific: $NWU = (1 - HU_{lesion}/HU_{contra})\times 100$ | Encodes known physical property of ischemic tissue |
| 4 | **Bilateral Symmetry Loss** | Anatomical prior: $L = \mathbb{E}[p \cdot ReLU(I_{mirror} - I)]$ | Exploits brain symmetry — strongest validated NCCT prior |

**References:**
- Chen et al. "Learning Euler's Elastica Model" — IEEE TMI 2022
- Kervadec et al. "Boundary loss for highly unbalanced segmentation" — MIDL 2019 / MedIA 2021
- Broocks et al. "Net water uptake in acute ischemic stroke" — 2022–2025
- Ni et al. "Asymmetry Disentanglement Network" — MICCAI 2022

### 1. Import Library

In [ ]:
!pip install -q segmentation-models-pytorch albumentations

In [ ]:
import gdown, zipfile, os, shutil, random, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import matplotlib.pyplot as plt
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp

### 2. Persiapan Dataset

In [ ]:
file_id = '1o0b6Nqs89zYoyRcnih5oGS7sk0bIrImo'
url = f'https://drive.google.com/uc?id={file_id}'
output_zip = 'dataset.zip'
print("Mengunduh dataset...")
gdown.download(url, output_zip, quiet=False)

In [ ]:
extract_dir = 'dataset'
print("Mengekstrak dataset...")
os.makedirs(extract_dir, exist_ok=True)
with zipfile.ZipFile(output_zip, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

### 2.b. Analisis Imbalance Data

In [ ]:
def analyze_class_imbalance(base_dir):
    mask_dir = os.path.join(base_dir, 'train', 'masks')
    mask_files = sorted([os.path.join(mask_dir, f) for f in os.listdir(mask_dir)])
    total_pixels = 0
    total_white_pixels = 0
    print(f"Menganalisis {len(mask_files)} mask...")
    for mask_path in mask_files:
        arr = np.array(Image.open(mask_path).convert('L'))
        binary = (arr > 127).astype(np.uint8) if arr.max() > 1 else (arr > 0.5).astype(np.uint8)
        total_pixels += binary.size
        total_white_pixels += np.sum(binary)
    w = (total_white_pixels/total_pixels)*100
    b = 100.0-w
    print(f"\nLesi: {total_white_pixels:,} ({w:.4f}%)")
    print(f"BG:   {total_pixels-total_white_pixels:,} ({b:.4f}%)")
    print(f"Rasio BG:Lesi = {b/w:.1f}:1")
    plt.figure(figsize=(6,4))
    bars = plt.bar(['Background','Lesion'],[b,w],color=['#34495e','#e74c3c'])
    plt.ylabel('%'); plt.title('Class Distribution')
    for bar in bars:
        y = bar.get_height()
        plt.text(bar.get_x()+bar.get_width()/2,y+1,f"{y:.2f}%",ha='center',fontweight='bold')
    plt.ylim(0,110); plt.show()
analyze_class_imbalance('dataset')

### 3. Dataset V8 — Enhanced Physics-Aware Dataset

**Signed Distance Transform (SDT) Maps:**

 SDT adalah peta jarak bertanda dari setiap piksel ke boundary ground truth mask. Piksel *di dalam* lesi → nilai positif (semakin ke tengah semakin besar). Piksel *di luar* lesi → nilai negatif (semakin jauh semakin negatif). Pikpersis di boundary → nilai 0.

SDT digunakan oleh **Boundary Loss** (Kervadec et al.). Dengan mengalikan probabilitas prediksi dengan SDT, loss bisa menghitung seberapa jauh prediksi *melenceng* dari boundary GT — tanpa harus menghitung Hausdorff Distance eksplisit yang tidak differensiabel. Karena SDT dihitung sekali di awal (pre-computed), tidak ada overhead komputasi saat training.

Model belajar menempatkan boundary dengan presisi tinggi. Loss ini *invariant terhadap class imbalance* karena hanya aktif di sekitar kontur — tidak peduli seberapa kecil lesinya, kontribusi loss tetap proporsional.

---
**Raw Image Tensors:**

Salinan intensitas piksel asli (sebelum augmentasi intensitas) yang disimpan bersama setiap sampel. Di V8, gambar dinormalisasi ke rentang [0,1] oleh `ToTensorV2`, dan `raw_img` adalah klon dari tensor tersebut.

**Fungsi di V8:** `raw_img` digunakan oleh dua loss berbasis fisika:
1. **Net Water Uptake (rHU) Loss** — membandingkan intensitas rata-rata di region prediksi vs region yang sama di hemisfer kontralateral (mirror). Lesi iskemik lebih gelap (hipodens) dengan NWU ~8%.
2. **Bilateral Symmetry Loss** — menghitung peta asimetri `ReLU(I_mirror - I)` untuk mendeteksi area yang lebih gelap dari sisi kontralateral (kandidat lesi).

**Mengapa tidak pakai HU asli?** Dataset berupa PNG (bukan DICOM), jadi nilai HU asli tidak tersedia. Intensitas [0,1] dari `ToTensorV2` sudah cukup sebagai proksi — rasio kecerahan antar hemisfer tetap terjaga meskipun nilai absolutnya berbeda dari HU sebenarnya.
Menambahkan Signed Distance Transform maps (untuk Boundary Loss) dan raw image tensors (untuk rHU/NWU Loss).

In [ ]:
from scipy.ndimage import distance_transform_edt

def compute_signed_distance_transform(mask_np, normalize=True):
    """Signed distance: positive inside GT, negative outside."""
    mask_int = (mask_np > 0.5).astype(np.uint8)
    sdt = distance_transform_edt(mask_int) - distance_transform_edt(1 - mask_int)
    if normalize:
        sdt = np.clip(sdt / (max(mask_np.shape)/2.0), -1.0, 1.0)
    return sdt

class NCCTDatasetV8(Dataset):
    """Returns 5-tuple: (img, mask, dist_map, label, raw_img)."""
    def __init__(self, split_dir, limit_size=None, transform=None, return_raw=True):
        self.split_dir = split_dir
        self.transform = transform
        self.return_raw = return_raw
        img_dir = os.path.join(split_dir, 'images')
        mask_dir = os.path.join(split_dir, 'masks')
        self.image_paths = sorted(os.path.join(img_dir, f) for f in os.listdir(img_dir))
        self.mask_paths = sorted(os.path.join(mask_dir, f) for f in os.listdir(mask_dir))
        if limit_size is not None:
            self.image_paths = self.image_paths[:limit_size]
            self.mask_paths = self.mask_paths[:limit_size]
    def __len__(self):
        return len(self.image_paths)
    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert('L')
        mask = Image.open(self.mask_paths[idx]).convert('L')
        if self.transform is not None:
            aug = self.transform(image=np.array(img).astype(np.uint8), mask=np.array(mask).astype(np.uint8))
            img_t = aug['image'].float() / 255.0
            mask_t = aug['mask'].float() / 255.0
        else:
            img_t = torch.from_numpy(np.array(img).astype(np.float32)).unsqueeze(0)/255.0
            mask_t = torch.from_numpy(np.array(mask).astype(np.float32)).unsqueeze(0)/255.0
        mask_bin = (mask_t.cpu().numpy() > 0.5).astype(np.float32)
        mask_t = torch.from_numpy(mask_bin).float()
        if mask_t.ndim == 2: mask_t = mask_t.unsqueeze(0)
        sdt = compute_signed_distance_transform(mask_bin.squeeze(), normalize=True)
        dist_t = torch.from_numpy(sdt).unsqueeze(0).float()
        raw = img_t.clone() if self.return_raw else torch.tensor([[0.0]])
        label = torch.tensor([1.0 if mask_t.max()>0 else 0.0], dtype=torch.float32)
        return img_t, mask_t, dist_t, label, raw

def get_dataloaders_v8(base_dir, batch_size=4, limit_size=None):
    trn_tfm = A.Compose([A.Resize(256,256), A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.1,scale_limit=0.1,rotate_limit=15,p=0.5), ToTensorV2()])
    val_tfm = A.Compose([A.Resize(256,256), ToTensorV2()])
    trn = NCCTDatasetV8(os.path.join(base_dir,'train'),limit_size,trn_tfm)
    val = NCCTDatasetV8(os.path.join(base_dir,'val'),limit_size,val_tfm)
    tst = NCCTDatasetV8(os.path.join(base_dir,'test'),limit_size,val_tfm)
    return (DataLoader(trn,batch_size,shuffle=True), DataLoader(val,batch_size,shuffle=False),
            DataLoader(tst,batch_size,shuffle=False))

def viz_sample(imgs,masks,dists,raws,n=3):
    n = min(n,imgs.shape[0])
    fig,axes = plt.subplots(n,4,figsize=(16,4*n))
    if n==1: axes=axes.reshape(1,-1)
    for i in range(n):
        axes[i,0].imshow(imgs[i,0].cpu().numpy(),cmap='gray'); axes[i,0].set_title('Input')
        axes[i,1].imshow(masks[i,0].cpu().numpy(),cmap='gray'); axes[i,1].set_title('GT')
        axes[i,2].imshow(dists[i,0].cpu().numpy(),cmap='RdBu',vmin=-1,vmax=1); axes[i,2].set_title('Signed Dist')
        axes[i,3].imshow(raws[i,0].cpu().numpy(),cmap='gray'); axes[i,3].set_title('Raw')
        for j in range(4): axes[i,j].axis('off')
    plt.tight_layout(); plt.show()

In [ ]:
train_loader,val_loader,test_loader=get_dataloaders_v8('dataset')
imgs,msks,dists,labs,raws=next(iter(val_loader))
print(f"Image: {imgs.shape}  Mask: {msks.shape}  Dist: {dists.shape}  [{dists.min():.2f},{dists.max():.2f}]")
viz_sample(imgs,msks,dists,raws,n=3)

### 4. Model Arsitektur

In [ ]:
class CSPBlock(nn.Module):
    def __init__(self,in_ch,out_ch):
        super().__init__()
        mid=out_ch//2
        self.main=nn.Sequential(nn.Conv2d(in_ch,mid,1,bias=False),nn.BatchNorm2d(mid),nn.ReLU(True),
                                nn.Conv2d(mid,mid,3,padding=1,bias=False),nn.BatchNorm2d(mid),nn.ReLU(True))
        self.skip=nn.Sequential(nn.Conv2d(in_ch,mid,1,bias=False),nn.BatchNorm2d(mid),nn.ReLU(True))
        self.trans=nn.Sequential(nn.Conv2d(out_ch,out_ch,1,bias=False),nn.BatchNorm2d(out_ch),nn.ReLU(True))
    def forward(self,x): return self.trans(torch.cat([self.main(x),self.skip(x)],dim=1))

class NCCTSegModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.c0=nn.Conv2d(1,32,3,padding=1)
        self.l1,l2,l3=CSPBlock(32,64),CSPBlock(64,128),CSPBlock(128,256)
        self.pool=nn.MaxPool2d(2,2)
        self.u1,u2,u3=nn.ConvTranspose2d(256,128,2,2),nn.ConvTranspose2d(128,64,2,2),nn.ConvTranspose2d(64,32,2,2)
        self.d1,d2,d3=CSPBlock(128+256,128),CSPBlock(64+128,64),CSPBlock(32+64,32)
        self.out=nn.Conv2d(32,1,1)
    def forward(self,x):
        x0=F.relu(self.c0(x))
        x1=self.l1(x0); p1=self.pool(x1)
        x2=self.l2(p1); p2=self.pool(x2)
        x3=self.l3(p2); f=self.pool(x3)
        d=self.d1(torch.cat([self.u1(f),x3],1))
        d=self.d2(torch.cat([self.u2(d),x2],1))
        d=self.d3(torch.cat([self.u3(d),x1],1))
        return self.out(d)

class UNet(nn.Module):
    def __init__(self,in_ch=1,out_ch=1):
        super().__init__()
        self.m=smp.Unet('resnet34',encoder_weights='imagenet',in_channels=in_ch,classes=out_ch,activation=None)
    def forward(self,x): return self.m(x)

class UNet_EffNet(nn.Module):
    def __init__(self,in_ch=1,out_ch=1):
        super().__init__()
        self.m=smp.Unet('efficientnet-b3',encoder_weights='imagenet',in_channels=in_ch,classes=out_ch,activation=None)
    def forward(self,x): return self.m(x)

class FPN_ResNext(nn.Module):
    def __init__(self,in_ch=1,out_ch=1):
        super().__init__()
        self.m=smp.FPN('resnext50_32x4d',encoder_weights='imagenet',in_channels=in_ch,classes=out_ch,activation=None)
    def forward(self,x): return self.m(x)


---
### 5. Enhanced Physics-Informed Loss Functions
V8 menggantikan TV Loss dengan 4 loss functions spesifik-domain.

#### 5.a. ACE Loss (Euler's Elastica) — Pengganti TV

Regularisasi smoothness yang memiliki sifat curvature-aware. Berbeda dengan TV loss yang hanya menghitung `|∇u|` (besarnya gradien), ACE menambahkan satu faktor lain yaitu suku kelengkungan κ². Boundary yang bergelombang (κ tinggi) akan dihukum lebih berat. Boundary yang lurus/tajam (κ rendah) tetap dipertahankan.

Fungsi loss ini untuk menghasilkan segmentasi yang lebih halus secara visual dimana boundary lesi tetap tajam dan tidak over-smoothed seperti efek TV. Cocok untuk lesi stroke yang punya batas relatif tegas (loss of gray-white differentiation).

**Referensi:** Chen et al., "Learning Euler's Elastica Model for Medical Image Segmentation" — IEEE TMI 2022
$$L_{ACE} = (\alpha + \beta\kappa^2)|\nabla u|$$
Kelengkungan $\kappa$ didekati dengan Laplacian. Tepi tajam dipertahankan, boundary bergelombang dihukum.

In [ ]:
class ACELoss(nn.Module):
    def __init__(self,alpha=1.0,beta=1.0):
        super().__init__(); self.alpha,self.beta=alpha,beta
    def forward(self,logits):
        p=torch.sigmoid(logits)
        gx,gy=torch.abs(p[:,:,:,1:]-p[:,:,:,:-1]).mean(),torch.abs(p[:,:,1:,:]-p[:,:,:-1,:]).mean()
        lx=torch.abs(p[:,:,:,2:]+p[:,:,:,:-2]-2*p[:,:,:,1:-1]).mean()
        ly=torch.abs(p[:,:,2:,:]+p[:,:,:-2,:]-2*p[:,:,1:-1,:]).mean()
        return (self.alpha+self.beta*((lx+ly)/2)**2)*((gx+gy)/2)

In [ ]:
ace=ACELoss()
s=torch.full((1,1,50,50),2.0)
n=torch.randn(1,1,50,50)*5.0
e=torch.cat([torch.full((1,1,50,25),3.0),torch.full((1,1,50,25),-3.0)],dim=-1)
print(f"Smooth: {ace(s):.4f}  Noisy: {ace(n):.4f} (HIGH)  SharpEdge: {ace(e):.4f} (LOW)")

#### 5.b. Boundary Loss (Kervadec et al.)

Loss ini membandingkan probabilitas prediksi dengan *Signed Distance Transform (SDT)* dari ground truth. `L_BL = Σ p · D_G` dimana `D_G` positif di luar boundary GT, negatif di dalam.

Loss hanya aktif di *sekitar boundary*. Ini membuatnya *invariant terhadap class imbalance* karena tidak dihitung per-pixel di seluruh area, melainkan hanya terkonsentrasi di kontur.

Boundary prediksi lebih presisi sehingga Hausdorff Distance (jarak terbesar antara boundary prediksi dan GT) menurun drastis.

**Referensi:** Kervadec et al., "Boundary loss for highly unbalanced segmentation" — MIDL 2019 / MedIA 2021. Teruji langsung pada ISLES (stroke segmentation).
$$L_{BL} = \frac{1}{N}\sum p_i \cdot D_{G,i}$$
Menggunakan pre-computed signed distance map. Positif jika prediksi keluar boundary GT.

In [ ]:
class BoundaryLoss(nn.Module):
    def forward(self,logits,dist_maps):
        return (torch.sigmoid(logits)*dist_maps).sum(dim=(2,3)).mean()

In [ ]:
bl=BoundaryLoss()
gy,gx=torch.meshgrid(torch.linspace(-1,1,50),torch.linspace(-1,1,50),indexing='ij')
gt=(gx**2+gy**2<0.5**2).float().unsqueeze(0).unsqueeze(0)
sdt=torch.from_numpy(compute_signed_distance_transform(gt[0,0].numpy(),True)).unsqueeze(0).unsqueeze(0).float()
print(f"Perfect:{bl(gt*5.0-(1-gt)*5.0,sdt):.4f}  Over:{bl(gt*5.0-(1-gt)*2.0,sdt):.4f}(HIGH)  Under:{bl(gt*2.0-(1-gt)*5.0,sdt):.4f}(NEG)")

#### 5.c. Net Water Uptake Loss (rHU/NWU)

Loss ini memanfaatkan *fisika spesifik CT* — lesi iskemik mengalami edema sitotoksik → penyerapan air → penurunan densitas (HU drop ~8). Rumus: `NWU = (1 - HU_lesion/HU_contra) × 100`. Loss menghukum prediksi dengan NWU yang tidak sesuai target ~8%.

Loss ini membandingkan intensitas rata-rata di region yang diprediksi sebagai lesi vs region yang sama di hemisfer kontralateral (mirror). Jika NWU tidak ~8%, maka prediksi tersebut tidak konsisten dengan fisika jaringan iskemik.

Mengurangi false positive karena prediksi harus konsisten dengan properti fisik CT (lesi harus lebih hipodens dari jaringan sehat di sisi lain).

**Referensi:** Broocks et al., "Net water uptake in acute ischemic stroke" — multiple publikasi 2022–2025
$$NWU = (1 - HU_{lesion}/HU_{contra})\times 100$$
Membandingkan intensitas lesi vs kontralateral. NWU ~8% untuk infark akut.

In [ ]:
class NetWaterUptakeLoss(nn.Module):
    def __init__(self,target_nwu=8.0):
        super().__init__(); self.target_nwu=target_nwu
    def forward(self,logits,raw):
        p=torch.sigmoid(logits)
        hl=(p*raw).sum(dim=(2,3))/(p.sum(dim=(2,3))+1e-8)
        hc=(torch.flip(p,dims=[-1])*torch.flip(raw,dims=[-1])).sum(dim=(2,3))/(torch.flip(p,dims=[-1]).sum(dim=(2,3))+1e-8)
        return torch.abs((1.0-hl/(hc+1e-8))*100.0-self.target_nwu).mean()

#### 5.d. Bilateral Symmetry Loss

Loss ini memanfaatkan *simetri bilateral otak* sebagai prior anatomis. `L_Sym = mean(p · ReLU(I_mirror - I))`. Jika tidak ada perbedaan antara hemisfer kiri dan kanan (asym_map ≈ 0), maka prediksi lesi di area tersebut akan dihukum.

Loss hanya mengizinkan prediksi di area yang menunjukkan asimetri (lebih gelap dari sisi kontralateral).

Secara dramatis mengurangi false positive karena model tidak bisa asal memprediksi lesi di area yang simetris.

**Referensi:** Ni et al., "Asymmetry Disentanglement Network for Interpretable AIS Infarct Segmentation in NCCT" — MICCAI 2022 (SOTA di AISD)
$$L_{Sym} = \mathbb{E}[p \cdot ReLU(I_{mirror} - I)]$$
Prediksi dihukum jika tidak ada asimetri (lesi hipodens memutus simetri otak).

In [ ]:
class BilateralSymmetryLoss(nn.Module):
    def forward(self,logits,raw):
        return (torch.sigmoid(logits)*(1.0-F.relu(torch.flip(raw,dims=[-1])-raw))).mean()

#### 5.e. Combined Loss



In [ ]:
class PhysicsEnhancedNCCTLoss(nn.Module):
    def __init__(self,bce_weight=0.5,gamma=2.0,pos_weight=66.0,
                 alpha_t=0.3,beta_t=0.7,la=0.01,lb=0.1,ln=0.01,ls=0.05,tn=8.0):
        super().__init__()
        self.w,self.g,self.pw,self.at,self.bt,self.la,self.lb,self.ln,self.ls = (
            bce_weight,gamma,pos_weight,alpha_t,beta_t,la,lb,ln,ls)
        self.ace,self.bnd,self.nwu,self.sym = ACELoss(),BoundaryLoss(),NetWaterUptakeLoss(tn),BilateralSymmetryLoss()
    def forward(self,logits,targets,dmaps,raw):
        pw=torch.tensor([self.pw],device=logits.device)
        bce=F.binary_cross_entropy_with_logits(logits,targets,pos_weight=pw,reduction='none')
        focal=((1-torch.exp(-bce))**self.g*bce).mean()
        p=torch.sigmoid(logits)
        inter=(p*targets).sum(dim=(2,3))
        fp,fn=(p*(1-targets)).sum(dim=(2,3)),((1-p)*targets).sum(dim=(2,3))
        tv=1.0-(1.0-(inter+1e-6)/(inter+self.at*fp+self.bt*fn+1e-6)).mean()
        base=self.w*focal+(1-self.w)*tv
        return base+self.la*self.ace(logits)+self.lb*self.bnd(logits,dmaps)+self.ln*self.nwu(logits,raw)+self.ls*self.sym(logits,raw)

class BaselineNCCTLoss(nn.Module):
    def __init__(self,bce_weight=0.5,gamma=2.0,pos_weight=66.0,alpha=0.3,beta=0.7):
        super().__init__(); self.w,self.g,self.pw,self.a,self.b=bce_weight,gamma,pos_weight,alpha,beta
    def forward(self,logits,targets):
        pw=torch.tensor([self.pw],device=logits.device)
        bce=F.binary_cross_entropy_with_logits(logits,targets,pos_weight=pw,reduction='none')
        focal=((1-torch.exp(-bce))**self.g*bce).mean()
        p=torch.sigmoid(logits)
        inter=(p*targets).sum(dim=(2,3))
        fp,fn=(p*(1-targets)).sum(dim=(2,3)),((1-p)*targets).sum(dim=(2,3))
        tv=1.0-(1.0-(inter+1e-6)/(inter+self.a*fp+self.b*fn+1e-6)).mean()
        return self.w*focal+(1-self.w)*tv

---
### 6. Training & Evaluation Functions

In [ ]:
def dice_coeff(pred,target,thresh=0.5,eps=1e-6):
    p=(torch.sigmoid(pred)>thresh).float(); t=(target>0.5).float()
    return (2*(p*t).sum(dim=(2,3))+eps).mean().item()/(p.sum(dim=(2,3))+t.sum(dim=(2,3))+eps).mean().item()

def train_model_v8(model,train_loader,val_loader,epochs=40,name='Model',patience=10,md=0.001,mode='enhanced'):
    opt=torch.optim.Adam(model.parameters(),lr=1e-4)
    sched=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode='min',factor=0.5,patience=5)
    if mode=='enhanced':
        crit=PhysicsEnhancedNCCTLoss(bce_weight=0.5,la=0.01,lb=0.1,ln=0.01,ls=0.05).cuda()
        extra=True; tag='V8-Enhanced'
    else:
        crit=BaselineNCCTLoss(bce_weight=0.5).cuda()
        extra=False; tag='V8-Baseline'
    print(f'\nTraining {name} [{tag}]...')
    print(f"{'Ep':<4}|{'T-Loss':<8}|{'V-Loss':<8}|{'V-Dice':<8}|{'LR':<10}|{'Time':<7}")
    print('-'*50)
    best_dice,best_vl,pc=0,float('inf'),0
    hist={'train_loss':[],'val_loss':[],'val_dice':[]}
    for ep in range(epochs):
        t0=time.time(); model.train(); tl=0
        for img,msk,dm,_,rw in train_loader:
            img,msk,dm,rw=[x.cuda() for x in (img,msk,dm,rw)]
            opt.zero_grad()
            l=crit(model(img),msk,dm,rw) if extra else crit(model(img),msk)
            l.backward(); opt.step(); tl+=l.item()
        model.eval(); vl,vd=0,0
        with torch.no_grad():
            for img,msk,dm,_,rw in val_loader:
                img,msk,dm,rw=[x.cuda() for x in (img,msk,dm,rw)]
                o=model(img)
                l=crit(o,msk,dm,rw) if extra else crit(o,msk)
                vl+=l.item(); vd+=dice_coeff(o,msk)
        dt=time.time()-t0
        atl,avl,avd=tl/len(train_loader),vl/len(val_loader),vd/len(val_loader)
        lr=opt.param_groups[0]['lr']
        hist['train_loss'].append(atl); hist['val_loss'].append(avl); hist['val_dice'].append(avd)
        sched.step(avl)
        print(f"{ep+1:<4}|{atl:<8.4f}|{avl:<8.4f}|{avd:<8.4f}|{lr:<10.6f}|{dt:<7.2f}s")
        if avd>best_dice:
            best_dice=avd; torch.save(model.state_dict(),f"{name.lower().replace(' ','_')}_{tag.lower()}_best.pth")
        if avl<best_vl-md: best_vl,pc=avl,0
        else: pc+=1
        if pc>=patience: print(f'Early stop at {ep+1}'); break
    return model,hist

def eval_model(models,test_loader):
    print(f"\n{'Model':<30}|{'Dice':<8}"); print('-'*38)
    res={}
    for n,m in models.items():
        m.eval(); d=0
        with torch.no_grad():
            for img,msk,_,_,_ in test_loader:
                d+=dice_coeff(m(img.cuda()),msk.cuda())
        res[n]=d/len(test_loader); print(f"{n:<30}|{res[n]:<8.4f}")
    return res

### 7. Inisialisasi

In [ ]:
train_loader,val_loader,test_loader=get_dataloaders_v8('dataset')
enh_models={}; base_models={}; histories={}

---
### 8. Training & Ablation Study
Enhanced (ACE+Boundary+NWU+Symmetry) vs Baseline (Focal+Tversky only).

#### 8.a. UNet (ResNet34)

In [ ]:
n='UNet_ResNet34'
me,he=train_model_v8(UNet().cuda(),train_loader,val_loader,epochs=5,name=f'{n}_V8-Enhanced',mode='enhanced')
enh_models[n]=me; histories[f'{n}_V8-Enhanced']=he

In [ ]:
mb,hb=train_model_v8(UNet().cuda(),train_loader,val_loader,epochs=5,name=f'{n}_V8-Baseline',mode='baseline')
base_models[n]=mb; histories[f'{n}_V8-Baseline']=hb

#### 8.b. UNet EfficientNet-b3

In [ ]:
n='UNet_EffNet'
me,he=train_model_v8(UNet_EffNet().cuda(),train_loader,val_loader,epochs=5,name=f'{n}_V8-Enhanced',mode='enhanced')
enh_models[n]=me; histories[f'{n}_V8-Enhanced']=he

In [ ]:
mb,hb=train_model_v8(UNet_EffNet().cuda(),train_loader,val_loader,epochs=5,name=f'{n}_V8-Baseline',mode='baseline')
base_models[n]=mb; histories[f'{n}_V8-Baseline']=hb

#### 8.c. FPN ResNeXt50

In [ ]:
n='FPN_ResNeXt'
me,he=train_model_v8(FPN_ResNext().cuda(),train_loader,val_loader,epochs=5,name=f'{n}_V8-Enhanced',mode='enhanced')
enh_models[n]=me; histories[f'{n}_V8-Enhanced']=he

In [ ]:
mb,hb=train_model_v8(FPN_ResNext().cuda(),train_loader,val_loader,epochs=5,name=f'{n}_V8-Baseline',mode='baseline')
base_models[n]=mb; histories[f'{n}_V8-Baseline']=hb

### 9. Evaluasi Ablation

In [ ]:
print('='*70)
print('            ABLATION: V8-Enhanced vs V8-Baseline')
print('='*70)
print('\n--- V8 Enhanced ---'); r_enh=eval_model(enh_models,test_loader)
print('\n--- V8 Baseline ---'); r_base=eval_model(base_models,test_loader)
print('\n'+'='*70); print('           PERBANDINGAN'); print('='*70)
print(f"\n{'Model':<25}|{'Enhanced':<10}|{'Baseline':<10}|{'Gain':<8}")
print('-'*58)
tg,ct=0,0
for m in r_enh:
    e,b=r_enh[m]*100,r_base[m]*100; g=e-b
    print(f"{m:<25}|{e:<8.2f}%|{b:<8.2f}%|{g:+7.2f}")
    tg+=g; ct+=1
if ct: print('-'*58); print(f"{'Rata-rata':<25}|{'':8}|{'':8}|{tg/ct:+7.2f}")
print('\n'+'='*70)
if tg>0: print('  >>> V8 Enhanced MENINGKATKAN performa! <<<'); print('  >>> ACE+Boundary+NWU+Symmetry > TV <<<')
else: print('  >>> Efek netral. Lambda perlu tuning. <<<')
print('='*70)

### 10. Visualisasi

In [ ]:
def vis_ablation(em,bm,tl,n=3):
    names=list(em.keys()); cols=2+len(names)*2
    plt.figure(figsize=(cols*3,n*3.5))
    with torch.no_grad():
        for i in range(n):
            img,msk,_,_,_=next(iter(tl)); g=img.cuda(); c=1
            plt.subplot(n,cols,i*cols+c); plt.imshow(img[0,0].cpu().numpy(),cmap='gray'); plt.title('Input'); plt.axis('off')
            c+=1; plt.subplot(n,cols,i*cols+c); plt.imshow(msk[0,0].cpu().numpy(),cmap='gray'); plt.title('GT'); plt.axis('off')
            for j,nm in enumerate(names):
                c+=1
                pb=(torch.sigmoid(bm[nm](g))>0.5).cpu().numpy()[0,0]
                plt.subplot(n,cols,i*cols+c); plt.imshow(pb,cmap='gray'); plt.title(f'{nm}\nBase'); plt.axis('off')
                c+=1; pe=(torch.sigmoid(em[nm](g))>0.5).cpu().numpy()[0,0]
                plt.subplot(n,cols,i*cols+c); plt.imshow(pe,cmap='gray'); plt.title(f'{nm}\nEnh'); plt.axis('off')
    plt.tight_layout(); plt.show()
if enh_models: vis_ablation(enh_models,base_models,test_loader)

### 11. Training Curves

In [ ]:
def plot_curves(hist):
    ek=[k for k in hist if 'Enhanced' in k]; bk=[k for k in hist if 'Baseline' in k]
    fig,ax=plt.subplots(2,2,figsize=(14,10))
    for a,k,t in [(ax[0,0],'train_loss','Train Loss'),(ax[0,1],'val_loss','Val Loss'),(ax[1,0],'val_dice','Val Dice')]:
        for x in ek: a.plot(hist[x][k],'-',label=x,lw=2)
        for x in bk: a.plot(hist[x][k],'--',label=x,lw=2)
        a.set(xlabel='Epoch',ylabel=t,title=t); a.legend(fontsize=7); a.grid(alpha=0.3)
    ax[1,1].axis('off')
    plt.suptitle('Ablation: V8-Enhanced vs Baseline',fontsize=14,fontweight='bold')
    plt.tight_layout(); plt.show()
if histories: plot_curves(histories)

### 12. Hyperparameter Tuning — Lambda

In [ ]:
configs=[
    {'ace':0.01,'bnd':0.1,'nwu':0.01,'sym':0.05},
    {'ace':0.05,'bnd':0.1,'nwu':0.01,'sym':0.05},
    {'ace':0.01,'bnd':0.5,'nwu':0.01,'sym':0.05},
    {'ace':0.01,'bnd':0.1,'nwu':0.05,'sym':0.05},
    {'ace':0.01,'bnd':0.1,'nwu':0.01,'sym':0.1},
    {'ace':0.05,'bnd':0.5,'nwu':0.05,'sym':0.1},
]
tune={}
print('='*70); print('   Lambda Tuning (5 epoch each)'); print('='*70)
for i,cfg in enumerate(configs):
    print(f'\nConfig {i+1}: ACE={cfg["ace"]} BND={cfg["bnd"]} NWU={cfg["nwu"]} SYM={cfg["sym"]}')
    m=UNet().cuda()
    crit=PhysicsEnhancedNCCTLoss(bce_weight=0.5,la=cfg['ace'],lb=cfg['bnd'],ln=cfg['nwu'],ls=cfg['sym']).cuda()
    opt=torch.optim.Adam(m.parameters(),lr=1e-4)
    for _ in range(5):
        m.train()
        for img,msk,dm,_,rw in train_loader:
            img,msk,dm,rw=[x.cuda() for x in (img,msk,dm,rw)]
            opt.zero_grad(); crit(m(img),msk,dm,rw).backward(); opt.step()
    m.eval(); vd=0
    with torch.no_grad():
        for img,msk,_,_,_ in val_loader:
            vd+=dice_coeff(m(img.cuda()),msk.cuda())
    tune[i]={'dice':vd/len(val_loader),'cfg':cfg}
    print(f'   Dice: {tune[i]["dice"]:.4f}')
print('\n'+'='*70)
print(f"{'Cfg':<6}|{'ACE':<6}|{'BND':<6}|{'NWU':<6}|{'SYM':<6}|{'Dice':<8}"); print('-'*42)
bi=max(tune,key=lambda k:tune[k]['dice'])
for i,d in tune.items():
    c=d['cfg']
    print(f"{i:<6}|{c['ace']:<6}|{c['bnd']:<6}|{c['nwu']:<6}|{c['sym']:<6}|{d['dice']:<8.4f}{' <<<' if i==bi else ''}")
bc=tune[bi]['cfg']
print(f"\n>>> Best: ACE={bc['ace']} BND={bc['bnd']} NWU={bc['nwu']} SYM={bc['sym']}")
print('='*70)

---
### 13. Ringkasan & Kesimpulan

**Perbandingan V7 vs V8:**

| Aspek | V7 (TV Loss) | V8 (Enhanced Physics) |
|-------|-------------|----------------------|
| Smoothness | TV: $|\nabla u|$ (buta kelengkungan) | ACE: $(\alpha+\beta\kappa^2)|\nabla u|$ (sadar curvature) |
| Boundary | Tidak ada | Boundary Loss (Kervadec, distance transform) |
| CT Physics | Tidak ada | NWU/rHU Loss (HU drop ~8%, Broocks) |
| Anatomi | Tidak ada | Bilateral Symmetry Loss (ADN, Ni et al.) |
| Imbalance | Weighted Focal + Tversky | Weighted Focal + Tversky |

**Referensi:**
- Chen et al. "Learning Euler's Elastica Model" — IEEE TMI 2022
- Kervadec et al. "Boundary loss for highly unbalanced segmentation" — MIDL 2019 / MedIA 2021
- Broocks et al. "Net water uptake in acute ischemic stroke" — 2022–2025
- Ni et al. "Asymmetry Disentanglement Network" — MICCAI 2022
- Shit et al. "clDice — Topology-Preserving Loss" — CVPR 2021
- Ding et al. "PINN-EMFNet" — 2024 (V7 reference)